In [ ]:
# ──────────────────────────────────────────────────────────────────────
# D) K-Nearest Neighbors (KNN) Classifier
# ──────────────────────────────────────────────────────────────────────
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

import os, time
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import HistGradientBoostingClassifier
import matplotlib.pyplot as plt


# ─────────── Configurar uso de 8 núcleos ───────────
os.environ["OMP_NUM_THREADS"]      = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"]      = "2"

# ─────────── Step 1: Carga y creación del target ───────────
t0 = time.time()
df = pd.read_csv("todoscsvs_transformado.csv")
print(f"Loaded data: {df.shape} in {time.time()-t0:.1f}s")

measures = list("ABCDEFGHIJKLMNÑO")
df["target"] = df[measures].idxmax(axis=1)
df.loc[df[measures].sum(axis=1)==0, "target"] = "NoMeasure"
counts = df["target"].value_counts()
rare   = counts[counts < 100].index.tolist()
df["target_grouped"] = df["target"].apply(lambda x: "Other" if x in rare else x)
print("Class counts after grouping:", df["target_grouped"].value_counts().to_dict())

# ─────────── Step 2: Separar features y etiqueta ───────────
X_raw = df.drop(columns=measures + ["target", "target_grouped"])
y_raw = df["target_grouped"]
le    = LabelEncoder()
y     = le.fit_transform(y_raw)
print("Encoded classes:", dict(enumerate(le.classes_)))

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train/Test split: {X_train_raw.shape[0]} / {X_test_raw.shape[0]} rows")

# ─────────── Step 3: Imputación y encoding ───────────
# 3.1 Imputar numéricos con mediana
num_cols   = X_train_raw.select_dtypes(include=[np.number]).columns
medians    = X_train_raw[num_cols].median()
X_train_raw[num_cols] = X_train_raw[num_cols].fillna(medians)
X_test_raw[num_cols]  = X_test_raw[num_cols].fillna(medians)

# 3.2 Identificar categóricas y ordinal encode
cat_cols = X_train_raw.select_dtypes(include=["object","category"]).columns.tolist()
print("Categorical columns:", cat_cols)
for c in cat_cols:
    X_train_raw[c] = X_train_raw[c].fillna("Missing").astype(str)
    X_test_raw[c]  = X_test_raw[c].fillna("Missing").astype(str)

enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
if cat_cols:
    X_train_raw[cat_cols] = enc.fit_transform(X_train_raw[cat_cols])
    X_test_raw[cat_cols]  = enc.transform(X_test_raw[cat_cols])

X_train = X_train_raw.copy()
X_test  = X_test_raw.copy()
print(f"Features ready: {X_train.shape[1]} columns")

# ─────────── Step 4: Sample weights para balance ───────────
train_counts   = Counter(y_train)
total          = len(y_train)
sample_weight  = np.array([ total/train_counts[c] for c in y_train ])
print(f"Sample‐weight range: {sample_weight.min():.1f} – {sample_weight.max():.1f}")

def plot_confusion(y_true, y_pred, title, classes, cmap):
    cm   = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12,12))
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)
    disp.plot(
        ax=plt.gca(),
        cmap=cmap,
        values_format='d',
        xticks_rotation=45,
        include_values=True
    )
    plt.title(title, fontsize=16)
    plt.xlabel("Predicted label", fontsize=14)
    plt.ylabel("True label", fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout(pad=3.0)
    plt.show()

# ──────────────────────────────────────────────────────────────────────
# D) K-Nearest Neighbors (KNN) Classifier - Versión Corregida
# ──────────────────────────────────────────────────────────────────────
print("\n=== K-Nearest Neighbors Classifier (Corrected) ===")

# Crear pipeline con escalado 
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# Definir parámetros para probar
knn_params = [
    {'knn__n_neighbors': [3, 5, 7, 9],
     'knn__weights': ['uniform', 'distance'],  
     'knn__p': [1, 2]},
]

# Usar GridSearchCV para encontrar los mejores parámetros
for i, param_grid in enumerate(knn_params, 1):
    print(f"\nKNN config {i}: {param_grid}")
    t4 = time.time()
    
    # Configurar GridSearchCV con validación cruzada
    knn = GridSearchCV(
        estimator=knn_pipeline,
        param_grid=param_grid,
        cv=5,
        scoring='f1_weighted',  
        n_jobs=8,
        verbose=1
    )
    
    # Entrenar SIN sample_weight 
    knn.fit(X_train, y_train)
    
    print(f"Mejores parámetros: {knn.best_params_}")
    print(f"Trained in {time.time()-t4:.1f}s")
    
    # Evaluar en el conjunto de prueba
    preds = knn.predict(X_test)
    print(classification_report(y_test, preds, zero_division=0))
    plot_confusion(y_test, preds, f"KNN Conf{i} Confusion Matrix", le.classes_, cmap="Reds")

    # Versión con los mejores parámetros
    best_knn = KNeighborsClassifier(**knn.best_params_['knn'])
    best_knn.fit(X_train, y_train)
    preds_best = best_knn.predict(X_test)
    print("\nPerformance with best params only:")
    print(classification_report(y_test, preds_best, zero_division=0))

Loaded data: (328959, 85) in 1.7s
Class counts after grouping: {'NoMeasure': 162194, 'I': 122387, 'A': 29603, 'C': 7334, 'B': 3389, 'F': 1677, 'G': 766, 'J': 682, 'E': 248, 'H': 213, 'D': 189, 'Other': 173, 'L': 104}
Encoded classes: {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G', 7: 'H', 8: 'I', 9: 'J', 10: 'L', 11: 'NoMeasure', 12: 'Other'}
Train/Test split: 263167 / 65792 rows
Categorical columns: []
Features ready: 69 columns
Sample‐weight range: 2.0 – 3170.7

=== K-Nearest Neighbors Classifier (Corrected) ===

KNN config 1: {'knn__n_neighbors': [3, 5, 7, 9], 'knn__weights': ['uniform', 'distance'], 'knn__p': [1, 2]}
Fitting 5 folds for each of 16 candidates, totalling 80 fits
